# Job Run Ingestion (Pipeline 002)

## Overview

Scheduled notebook that polls the Databricks Jobs REST API and merges run results into `rag_jobs.jobs_run_log`.

### Pipeline Order
`001` (one-time setup) → **`002`** (scheduled) → `003` (embedding) → `004` (query)

## What This Notebook Does

- **Fetches Terminal Runs:** Polls Databricks Jobs API for all completed runs within a configurable lookback window
  - Widget parameter: `lookback_hours` - controls how far back to look (e.g., 1 hour)
  - Filters to terminal states: `TERMINATED`, `INTERNAL_ERROR`, `SKIPPED`

- **Extracts Error Details:** For failed runs:
  - Fetches the full traceback via the run-output API
  - Classifies the error type (e.g., `NullPointerException`, `OutOfMemoryError`)

- **Merges to Log Table:** Inserts results into `rag_jobs.jobs_run_log`
  - Insert-only logic: already-seen `run_id` / `job_id` pairs are skipped
  - No updates to existing records

## Scheduling

Run on a schedule that slightly overlaps your job cadence:
- Example: If jobs run every 30 minutes, run this notebook every 30 minutes with `lookback_hours = 1`
- This ensures no runs are missed even with occasional API latency

In [ ]:
# COMMAND ----------

# DBTITLE 1,Setup
# DBTITLE 1,Setup
import requests
from datetime import datetime, timezone

# Widget: overlap your schedule interval (e.g. 30-min schedule -> 1hr lookback)
dbutils.widgets.text("lookback_hours", "1", "Lookback hours")
LOOKBACK_HOURS = int(dbutils.widgets.get("lookback_hours"))

DATABRICKS_HOST  = spark.conf.get("spark.databricks.workspaceUrl")
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

headers  = {"Authorization": f"Bearer {DATABRICKS_TOKEN}"}
base_url = f"https://{DATABRICKS_HOST}/api/2.1/jobs"

print(f"Workspace : {DATABRICKS_HOST}")
print(f"Lookback  : {LOOKBACK_HOURS}h")


In [ ]:
# COMMAND ----------

# DBTITLE 1,Fetch recent job runs
# DBTITLE 1,Fetch recent job runs
start_time_ms = int((datetime.now(timezone.utc).timestamp() - LOOKBACK_HOURS * 3600) * 1000)

all_runs: list[dict] = []
page_token = None

while True:
    params: dict = {"start_time_from": start_time_ms, "limit": 25, "expand_tasks": "true"}
    if page_token:
        params["page_token"] = page_token

    try:
        resp = requests.get(f"{base_url}/runs/list", headers=headers, params=params, timeout=30)
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"Jobs API pagination failed: {e}") from e

    data = resp.json()
    all_runs.extend(data.get("runs", []))

    if not data.get("has_more"):
        break
    page_token = data.get("next_page_token")

print(f"\u2705 Fetched {len(all_runs)} job runs from the last {LOOKBACK_HOURS}h")


In [ ]:
# COMMAND ----------

# DBTITLE 1,Filter to terminal runs
# DBTITLE 1,Filter to terminal runs
TERMINAL_STATES = {"TERMINATED", "INTERNAL_ERROR", "SKIPPED"}

def extract_run_info(run: dict) -> dict | None:
    state            = run.get("state", {})
    life_cycle_state = state.get("life_cycle_state")

    if life_cycle_state not in TERMINAL_STATES:
        return None  # still running or queued - skip

    result_state = state.get("result_state", "UNKNOWN")  # SUCCESS, FAILED, TIMEDOUT, CANCELED
    status       = "SUCCESS" if result_state == "SUCCESS" else "FAILED"

    start_ts = run.get("start_time", 0) / 1000
    run_date = datetime.fromtimestamp(start_ts, tz=timezone.utc).date() if start_ts else None

    duration_secs = int(run.get("run_duration", 0) / 1000) if run.get("run_duration") else None

    cluster_id = (
        run.get("cluster_spec", {}).get("existing_cluster_id")
        or (run.get("tasks") or [{}])[0].get("existing_cluster_id")
        or "unknown"
    )

    return {
        "run_id":        run.get("run_id"),
        "job_id":        str(run.get("job_id", run.get("run_id"))),
        "job_name":      run.get("run_name", "unknown"),
        "run_date":      run_date,
        "status":        status,
        "cluster_id":    cluster_id,
        "triggered_by":  run.get("trigger", "unknown"),
        "duration_secs": duration_secs,
    }

run_records = [r for r in (extract_run_info(run) for run in all_runs) if r]

failed_count  = sum(1 for r in run_records if r["status"] == "FAILED")
success_count = sum(1 for r in run_records if r["status"] == "SUCCESS")
print(f"Terminal runs: {len(run_records)} total  ({success_count} succeeded, {failed_count} failed)")

In [ ]:
# COMMAND ----------

# DBTITLE 1,Fetch tracebacks for failed runs
# DBTITLE 1,Fetch tracebacks for failed runs
def get_traceback(run_id: int) -> str:
    try:
        resp = requests.get(
            f"{base_url}/runs/get-output",
            headers=headers,
            params={"run_id": run_id},
            timeout=30,
        )
        resp.raise_for_status()
        output = resp.json()
        return output.get("error_trace") or output.get("error") or "No traceback available"
    except Exception as e:
        return f"Failed to fetch traceback: {e}"

def extract_error_type(traceback: str) -> tuple[str, str]:
    """Return (error_type, short_error_message) from a traceback string."""
    if not traceback or traceback.startswith(("No traceback", "Failed to fetch")):
        return "UNKNOWN", traceback or ""

    known_types = [
        "NullPointerException", "AnalysisException", "OutOfMemoryError",
        "FileNotFoundException", "TimeoutException", "PermissionError",
        "ConnectionError", "ValueError", "KeyError", "AttributeError",
    ]
    for err_type in known_types:
        if err_type in traceback:
            for line in traceback.split("\n"):
                if err_type in line:
                    return err_type, line.strip()[:500]

    # No known type matched - use the last non-empty line as a fallback message
    lines = [line for line in traceback.split("\n") if line.strip()]
    return "UNKNOWN", (lines[-1].strip()[:500] if lines else "")

for record in run_records:
    if record["status"] == "FAILED":
        tb = get_traceback(record["run_id"])
        error_type, error_message  = extract_error_type(tb)
        record["traceback"]        = tb
        record["error_type"]       = error_type
        record["error_message"]    = error_message
    else:
        record["traceback"]     = None
        record["error_type"]    = None
        record["error_message"] = None

print(f"\u2705 Tracebacks fetched for {failed_count} failed runs")

In [ ]:
# COMMAND ----------

# DBTITLE 1,Merge into jobs_run_log
# DBTITLE 1,Merge into jobs_run_log
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, LongType

schema = StructType([
    StructField("run_id",        LongType(),    True),
    StructField("job_id",        StringType(),  True),
    StructField("job_name",      StringType(),  True),
    StructField("run_date",      DateType(),    True),
    StructField("status",        StringType(),  True),
    StructField("cluster_id",    StringType(),  True),
    StructField("triggered_by",  StringType(),  True),
    StructField("duration_secs", IntegerType(), True),
    StructField("traceback",     StringType(),  True),
    StructField("error_type",    StringType(),  True),
    StructField("error_message", StringType(),  True),
])

if run_records:
    new_runs_df = spark.createDataFrame(run_records, schema=schema)
    new_runs_df.createOrReplaceTempView("incoming_runs")

    spark.sql("""
        MERGE INTO rag_jobs.jobs_run_log AS target
        USING incoming_runs AS source
        ON target.job_id = source.job_id AND target.run_id = source.run_id
        WHEN NOT MATCHED THEN INSERT (
            run_id, job_id, job_name, run_date, status, cluster_id,
            triggered_by, traceback, error_message, error_type, duration_secs
        )
        VALUES (
            source.run_id, source.job_id, source.job_name, source.run_date, source.status,
            source.cluster_id, source.triggered_by, source.traceback, source.error_message,
            source.error_type, source.duration_secs
        )
    """)
    print(f"\u2705 Merged {len(run_records)} runs into jobs_run_log")
else:
    print("\u2139\ufe0f  No terminal runs in this window - nothing to merge")

In [ ]:
# COMMAND ----------

# DBTITLE 1,Summary
# DBTITLE 1,Summary
print("=== Recent entries in jobs_run_log ===")
spark.sql("""
    SELECT run_id, job_id, job_name, run_date, status, error_type, duration_secs
    FROM rag_jobs.jobs_run_log
    ORDER BY run_date DESC, run_id DESC
    LIMIT 20
""").show(truncate=False)

print("=== Unembedded failed runs (pending notebook 003) ===")
spark.sql("""
    SELECT run_id, job_id, job_name, run_date, error_type
    FROM rag_jobs.jobs_run_log
    WHERE status = 'FAILED' AND embedded_at IS NULL
    ORDER BY run_date DESC
""").show(truncate=False)